In [ ]:
#| default_exp pii


# pii

> A PII detector you fit on your own examples, and arithmetic where you have none.

An organisation's PII is mostly not the world's PII. `EMP-483920` identifies somebody and no pattern
bank ships it; `ORD-483920` identifies a pallet and has the same shape. A 16-digit device serial that
passes Luhn is a payment card to every detector that only knows arithmetic.

So this fits one. `fit(data)` takes whatever a team already labelled, learns a span tagger and a
document classifier over token shape, context and the baseline's own verdicts, and returns a
detector. With no examples to fit, `fit` returns a detector that is the baseline, so the fallback is
the same object rather than a different code path.

The arbitration rule is the point: a kind the model was trained on, the model decides, veto included.
A kind it never saw, the baseline passes through untouched.


In [ ]:
#| export
from __future__ import annotations
import csv, json, os, re, time
from pathlib import Path

import numpy as np
from fastcore.all import AttrDict, L, first, patch, store_attr


In [ ]:
#| hide
from fastcore.test import *
from tempfile import mkdtemp
FIX = Path('fixtures')


## What a labelled example is

Three fields, any of which a dataset may leave out: `text`, `spans` as `(start, end, kind)`, and a
document `label`. An example with no spans and no label is a negative, which is the convention every
span annotator follows.

`group` is the split unit. Two examples that share a group never land on opposite sides of
`Dataset.split`, so a template, a document a corpus was chunked from, or a near-duplicate cannot
leak across it.


In [ ]:
#| export
#: Document classes that mean nothing in here is anybody's business.
CLEAN = frozenset({'clean', 'none', 'no', 'negative', 'public', 'ok', 'false', '0', 'o'})
#: Where the text is, in the exports the annotation tools write.
TEXT_KEYS = ('text', 'content', 'body', 'document', 'doc', 'snippet')
#: Where the spans are. Checked before `LABEL_KEYS`, so doccano's `labels` is not read as a class.
SPAN_KEYS = ('spans', 'entities', 'labels', 'annotations', 'result', 'ents')
#: Where the document class is.
LABEL_KEYS = ('label', 'target', 'class', 'y', 'has_pii', 'pii', 'sensitivity')

def _get(d:dict, *ks):
    "The first of `ks` that `d` has a value for."
    for k in ks:
        if (v := d.get(k)) is not None: return v

def kind(k) -> str:
    "A span kind as a bare name: a BIO prefix comes off and the rest is lowercased to snake case."
    s = re.sub(r'^[BILUES]-(?=\S)', '', str('pii' if k is None else k).strip())
    return re.sub(r'[^a-z0-9]+', '_', s.lower()).strip('_') or 'pii'

def as_span(o) -> tuple:
    "One labelled span as `(start, end, kind)`, from a tuple, a doccano dict, or a Label Studio result."
    if isinstance(o, (tuple, list)):
        s, e, *rest = o
        return int(s), int(e), kind(first(rest))
    if not isinstance(o, dict): raise ValueError(f'not a span: {o!r}')
    d = o['value'] if isinstance(o.get('value'), dict) else o
    s, e = _get(d, 'start', 'start_offset', 'begin', 'char_start'), _get(d, 'end', 'end_offset', 'char_end')
    if s is None or e is None: raise ValueError(f'span carries no offsets: {o!r}')
    k = _get(d, 'kind', 'label', 'labels', 'entity', 'type', 'tag')
    return int(s), int(e), kind(first(k) if isinstance(k, (list, tuple, L)) else k)


In [ ]:
#| export
def example(o,                  # a dict, a `(text, spans)` or `(text, label)` pair, or bare text
            text_key:str=None,  # which field holds the text, when the guess is wrong
           ) -> AttrDict:
    "One labelled example: `text`, `spans` as `(start, end, kind)`, and a document `label`."
    if isinstance(o, str): d = {'text': o}
    elif isinstance(o, (tuple, list)):
        t, *rest = o
        d = {'text': t}
        if rest: d['spans' if isinstance(rest[0], (list, tuple, L)) else 'label'] = rest[0]
    elif isinstance(o, dict): d = dict(o)
    else: raise ValueError(f'not an example: {o!r}')
    t = _get(d, text_key) if text_key else _get(d, *TEXT_KEYS)
    txt = str('' if t is None else t)
    sp = []
    for k in SPAN_KEYS:
        v = d.get(k)
        if isinstance(v, (list, tuple, L)) and len(v) and isinstance(first(v), (list, tuple, dict)):
            sp = sorted(as_span(s) for s in v); break
    e = AttrDict(text=txt, spans=sp, label=_doc_label(_get(d, *LABEL_KEYS), sp))
    if (g := d.get('group')) is not None: e.group = g
    return e

def _doc_label(lab, spans) -> str:
    "The document class: what the dataset says, else `pii` when a span was marked and `clean` when none was."
    if lab is None or lab == '': return 'pii' if spans else 'clean'
    if isinstance(lab, (bool, int, float, np.integer)): return 'pii' if lab else 'clean'
    k = kind(lab)
    return 'clean' if k in CLEAN else k


In [ ]:
#| hide
test_eq(as_span((3, 9, 'B-EMP_ID')), (3, 9, 'emp_id'))
test_eq(as_span({'start_offset': 1, 'end_offset': 4, 'label': 'Staff Login'}), (1, 4, 'staff_login'))
test_eq(as_span({'value': {'start': 2, 'end': 5, 'labels': ['Badge']}}), (2, 5, 'badge'))
test_fail(lambda: as_span({'label': 'x'}), contains='no offsets')

_e = example({'text': 'badge B-1234 opened it', 'labels': [[6, 12, 'badge']]})
test_eq((_e.spans, _e.label), ([(6, 12, 'badge')], 'pii'))
test_eq(example('nothing here').label, 'clean')                    # no spans, no label: a negative
test_eq(example(('x', [])).label, 'clean')
test_eq(example({'text': 'x', 'has_pii': False}).label, 'clean')
test_eq(example({'content': 'x', 'sensitivity': 'Restricted'}).label, 'restricted')
test_eq(example(('x', 'clean')).spans, [])


## The dataset


In [ ]:
#| export
class Dataset(L):
    "Labelled examples, and the splits and counts a fit needs off them."
    @property
    def kinds(self) -> L:
        "Every span kind in the data, sorted."
        return L(sorted({k for e in self for _, _, k in e.spans}))
    @property
    def classes(self) -> L:
        "Every document class in the data, sorted."
        return L(sorted({e.label for e in self}))
    @property
    def positive(self) -> Dataset:
        "The examples that carry identity: a span, or a class that is not in `CLEAN`."
        return Dataset(e for e in self if e.spans or e.label not in CLEAN)

    def counts(self) -> AttrDict:
        "How many examples, spans per kind, and examples per class."
        ks, cs = {}, {}
        for e in self:
            cs[e.label] = cs.get(e.label, 0) + 1
            for _, _, k in e.spans: ks[k] = ks.get(k, 0) + 1
        return AttrDict(n=len(self), spans=dict(sorted(ks.items(), key=lambda t: -t[1])),
                        classes=dict(sorted(cs.items(), key=lambda t: -t[1])),
                        groups=len({_gkey(i, e) for i, e in enumerate(self)}))

    def split(self, valid:float=0.25, seed:int=0) -> tuple:
        "Train and validation, holding out whole `group`s so nothing leaks across the split."
        gs = list(dict.fromkeys(_gkey(i, e) for i, e in enumerate(self)))
        if not valid or len(gs) < 2: return self, Dataset()
        order = np.random.default_rng(seed).permutation(len(gs))
        hold = {gs[int(i)] for i in order[:max(1, round(valid*len(gs)))]}
        keys = [_gkey(i, e) for i, e in enumerate(self)]
        return (Dataset(e for e, k in zip(self, keys) if k not in hold),
                Dataset(e for e, k in zip(self, keys) if k in hold))

    def save(self, path) -> Path:
        "Write the examples as JSON lines."
        p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text('\n'.join(json.dumps({k: v for k, v in e.items()}) for e in self) + '\n')
        return p

    def _repr_markdown_(self):
        c = self.counts()
        return (f"{c.n} examples, {c.groups} groups\n\n"
                f"- classes: {c.classes}\n- spans: {c.spans}")

def _gkey(i, e):
    "What an example is split by: its `group` when it has one, else itself."
    return ('g', e['group']) if e.get('group') is not None else ('i', i)


In [ ]:
#| export
#: Text files a folder of examples is read from.
TEXT_EXTS = ('.txt', '.md', '.eml', '.log')

def dataset(src,                 # a `.jsonl`/`.json`/`.csv` file, a folder of `<class>/*.txt`, or examples
            text_key:str=None,   # which field holds the text, when the guess is wrong
           ) -> Dataset:
    "Examples from a file, a folder of class directories, or anything iterable."
    if isinstance(src, Dataset): return src
    if isinstance(src, (str, Path)):
        p = Path(src).expanduser()
        if p.is_dir(): return Dataset(_folder_examples(p))
        rows = _read_rows(p)
    else: rows = src
    return Dataset(example(o, text_key) for o in rows)

def _read_rows(p:Path) -> list:
    "Rows out of one file, by suffix."
    if not p.exists(): raise FileNotFoundError(p)
    if p.suffix == '.jsonl': return [json.loads(l) for l in p.read_text().splitlines() if l.strip()]
    if p.suffix == '.json':
        o = json.loads(p.read_text())
        return o if isinstance(o, list) else (o.get('examples') or o.get('data') or [])
    if p.suffix in ('.csv', '.tsv'):
        with p.open(newline='') as f:
            return list(csv.DictReader(f, delimiter='\t' if p.suffix == '.tsv' else ','))
    raise ValueError(f'cannot read examples from {p.suffix or p.name!r}: '
                     'expected .jsonl, .json, .csv, .tsv, or a folder of <class>/*.txt')

def _folder_examples(p:Path):
    "`<class>/<file>.txt` is one example of class `<class>`, grouped by file stem."
    for d in sorted(x for x in p.iterdir() if x.is_dir()):
        for f in sorted(x for x in d.rglob('*') if x.suffix.lower() in TEXT_EXTS):
            yield example({'text': f.read_text(errors='replace'), 'label': d.name, 'group': f.stem})


In [ ]:
#| hide
_ds = dataset(FIX/'pii_org.jsonl')
test_eq(len(_ds), 240)
test_eq(_ds.classes, ['clean', 'pii'])
test_eq('emp_id' in _ds.kinds, True)
_tr, _va = _ds.split(0.25, seed=0)
test_eq(len(_tr) + len(_va), len(_ds))
test_eq(len(_va) > 20, True)
# no group straddles the split, which is what makes a held-out score mean anything
test_eq(set(e.group for e in _tr) & set(e.group for e in _va), set())

_d = Path(mkdtemp()); (_d/'pii').mkdir(); (_d/'clean').mkdir()
(_d/'pii'/'a.txt').write_text('mail jane@example.com'); (_d/'clean'/'b.txt').write_text('nothing')
test_eq(dataset(_d).classes, ['clean', 'pii'])
test_eq(dataset([('a', [[0, 1, 'k']]), 'b']).kinds, ['k'])
test_fail(lambda: dataset(_d/'pii'/'a.txt'), contains='cannot read examples')


## Tokens, shapes and features

Letters, digits and punctuation split apart, so `EMP-483920` is three tokens and a shape feature can
see the prefix without the digits. Offsets are kept, because a span is characters and the model
works in tokens.

The features are the ordinary ones for a linear tagger: the token, its shape, a window of neighbours,
and what the baseline said about each of them. That last group is what lets a fitted model overrule
arithmetic instead of arguing with it.


In [ ]:
#| export
#: Letter runs, digit runs, and one character for anything else.
TOKEN = re.compile(r'[^\W\d_]+|\d+|[^\w\s]|_')

def tokens(text:str) -> list:
    "Tokens as `(start, end, text)`."
    return [(m.start(), m.end(), m[0]) for m in TOKEN.finditer(str(text or ''))]

def shape(t:str) -> str:
    "Letters as `X`/`x`, digits as `d`, everything else itself: `EMP` -> `XXX`, `4839` -> `dddd`."
    return ''.join('d' if c.isdigit() else 'X' if c.isupper() else 'x' if c.isalpha() else c for c in t)

def cshape(t:str) -> str:
    "`shape` with runs collapsed to two, so `483920` and `4839201` are one feature."
    return re.sub(r'(.)\1+', r'\1\1', shape(t))

def luhn(s:str) -> bool:
    "The check digit every payment card carries. Also the one Swedish and IMEI numbers use."
    ds = [int(c) for c in s if c.isdigit()]
    if len(ds) < 12: return False
    tot, parity = 0, len(ds) % 2
    for i, d in enumerate(ds):
        if i % 2 == parity: d = d*2 - 9 if d*2 > 9 else d*2
        tot += d
    return tot % 10 == 0


In [ ]:
#| export
def bio(toks:list, spans:list) -> list:
    "One tag per token: `O`, or `B-`/`I-` and the kind. A span that starts mid-token claims the token."
    tags = ['O']*len(toks)
    for s, e, k in sorted(spans):
        for j, i in enumerate([i for i, (ts, te, _) in enumerate(toks) if ts < e and s < te]):
            tags[i] = ('B-' if j == 0 else 'I-') + k
    return tags

def bio_spans(toks:list, tags:list) -> list:
    "The reverse: `(start, end, kind)` per run of tags, an `I-` continuing only its own kind."
    out = []
    for i, t in enumerate(tags):
        if t == 'O': continue
        p, k = t.split('-', 1)
        if p == 'B' or not out or out[-1][2] != k or out[-1][3] != i - 1: out.append([toks[i][0], toks[i][1], k, i])
        else: out[-1][1], out[-1][3] = toks[i][1], i
    return [(s, e, k) for s, e, k, _ in out]

def token_feats(toks:list,        # tokens from `tokens`
                i:int,            # which one
                base:list=None,   # per-token baseline kind, from `base_tags`
                window:int=2,     # tokens of context each side
               ) -> dict:
    "Features for one token: its own form and shape, its neighbours, and what the baseline said."
    n, w = len(toks), toks[i][2]
    f = {'w': w.lower(), 'sh': shape(w), 'csh': cshape(w), 'len': min(len(w), 12),
         'up': w.isupper(), 'ti': w.istitle(), 'dig': w.isdigit(), 'al': w.isalpha()}
    if w.isdigit(): f |= {'nd': min(len(w), 20), 'luhn': luhn(w)}
    if w.isalpha() and len(w) > 3: f |= {'pre3': w[:3].lower(), 'suf3': w[-3:].lower()}
    if base is not None: f['base'] = base[i] or '-'
    for d in range(1, window + 1):
        for j, tag in ((i - d, f'-{d}'), (i + d, f'+{d}')):
            if not 0 <= j < n: f[f'{tag}:w'] = '<bos>' if j < 0 else '<eos>'; continue
            f[f'{tag}:w'] = toks[j][2].lower()
            f[f'{tag}:sh'] = cshape(toks[j][2])
            if base is not None: f[f'{tag}:base'] = base[j] or '-'
    return f

def base_tags(toks:list, spans:list) -> list:
    "The baseline's kind for each token, `None` where it found nothing."
    out = [None]*len(toks)
    for s, e, k, *_ in spans:
        for i, (ts, te, _) in enumerate(toks):
            if ts < e and s < te: out[i] = k
    return out


In [ ]:
#| hide
test_eq([t[2] for t in tokens('EMP-483920 is x_y')], ['EMP', '-', '483920', 'is', 'x', '_', 'y'])
test_eq(shape('EMP-4839'), 'XXX-dddd')
test_eq(cshape('EMP-483920'), 'XX-dd')
test_eq(luhn('4556737586899855'), True)
test_eq(luhn('4556737586899856'), False)

_t = tokens('badge B-1234 opened')
test_eq(bio(_t, [(6, 12, 'badge')]), ['O', 'B-badge', 'I-badge', 'I-badge', 'O'])
test_eq(bio_spans(_t, bio(_t, [(6, 12, 'badge')])), [(6, 12, 'badge')])
# a B- tag always opens a span, so two of the same kind side by side stay two
test_eq(len(bio_spans(_t[:2] + _t[2:], ['O', 'B-k', 'B-k', 'O', 'O'])), 2)
test_eq(token_feats(_t, 1)['sh'], 'X')
test_eq(token_feats(_t, 0, base_tags(_t, [(6, 12, 'badge', 'B-1234')]))['+1:base'], 'badge')


## The baseline

`floor_spans` is a floor, not a detector: six patterns with nothing regional in them.
`vishalakshi.pii.pii_spans` is the real bank, 34 kinds with a checksum each, and it is used
automatically when it is installed. `base=` takes either, or any callable that returns
`(start, end, kind, text)`.


In [ ]:
#| export
#: kind -> (pattern, validator). What is left when there is no pattern bank installed.
FLOOR = {
    'email':  (r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', None),
    'card':   (r'\b[2-6](?:[ -]?\d){12,18}\b', luhn),
    'ssn':    (r'\b\d{3}-\d{2}-\d{4}\b', None),
    'ip':     (r'\b(?:(?:25[0-5]|2[0-4]\d|1?\d?\d)\.){3}(?:25[0-5]|2[0-4]\d|1?\d?\d)\b', None),
    'phone':  (r'\+\d{1,3}[ .-]?\(?\d{1,5}\)?[ .-]?\d{3,4}[ .-]?\d{3,4}\b|\b\d{3}-\d{3}-\d{4}\b', None),
    'secret': (r'\b(?:sk-[A-Za-z0-9_-]{16,}|ghp_[A-Za-z0-9]{20,}|AKIA[0-9A-Z]{16}|AIza[0-9A-Za-z_-]{35})\b', None),
}
_FLOOR = {k: (re.compile(p, re.I), v) for k, (p, v) in FLOOR.items()}

def no_overlap(spans:list) -> list:
    "Spans de-overlapped longest first, then leftmost. What every layer here returns."
    out, taken = [], []
    for sp in sorted(spans, key=lambda s: (s[0] - s[1], s[0])):
        if any(sp[0] < b and a < sp[1] for a, b in taken): continue
        taken.append((sp[0], sp[1])); out.append(sp)
    return sorted(out)

def floor_spans(text:str) -> list:
    "Spans from `FLOOR` as `(start, end, kind, text)`."
    text = str(text or '')
    return no_overlap([(m.start(), m.end(), k, m[0]) for k, (rx, ok) in _FLOOR.items()
                      for m in rx.finditer(text) if ok is None or ok(m[0])])

def vishalakshi_spans(text:str) -> list:
    "Spans from `vishalakshi.pii`, which is 34 kinds with a checksum each."
    from vishalakshi.pii import pii_spans
    return [tuple(s) for s in pii_spans(text)]

def baseline(fn=None) -> callable:
    "The fallback detector: `fn`, else vishalakshi's bank when it imports, else `floor_spans`."
    if fn is not None: return fn
    if os.environ.get('ANYA_PII_BASELINE') == 'floor': return floor_spans
    try:
        import vishalakshi.pii    # noqa: F401
        return vishalakshi_spans
    except Exception: return floor_spans


In [ ]:
#| hide
test_eq(floor_spans('mail jane@example.com now'), [(5, 21, 'email', 'jane@example.com')])
test_eq(floor_spans('card 4556737586899855.')[0][2], 'card')
test_eq(floor_spans('serial 4556737586899856.'), [])                # fails Luhn, so not a card
test_eq(len(floor_spans('EMP-483920 approved it')), 0)              # no pattern bank knows this
# the de-overlapper keeps the longer span when two kinds claim the same characters
test_eq(no_overlap([(0, 4, 'a', 'abcd'), (2, 6, 'b', 'cdef')]), [(0, 4, 'a', 'abcd')])


## The detector

One object in two states. Fitted, it tags tokens and classifies documents; unfitted, it is the
baseline. Nothing above it has to know which, which is what makes "no examples yet" the same code
path as "trained last week".

`mode` decides how the two layers combine:

| mode | learned spans | baseline spans |
|---|---|---|
| `defer` | only of kinds the baseline never produced | all kept |
| `union` | kept | all kept, longest wins an overlap |
| `hybrid` | kept | dropped for kinds the model was trained on |
| `learned` | kept | not consulted |
| `baseline` | not consulted | kept |

`defer` is the default, and the reason is a failure it prevents. A tagger trained on payment cards
learns that sixteen Luhn-valid digits are a card, and then labels device serials the pattern bank had
correctly gated: 10 invented spans over five splits of `evals/mkpii.py`, all of them `card`. Deferring
on the kinds the baseline can already produce removes every one of them, for 0.907 F1 at 1.000
precision against `union`'s 0.876 at 0.946, on the same recall.

`hybrid` is the other direction, and it is what a team whose pattern bank is the problem wants: it
lets the tagger suppress a kind it was trained on, which is how a labelled device serial stops being
a payment card. It only pays where the baseline is the weaker layer, so `evals/RESULTS.md` reports it
against both.

The document class decides `has_pii` only when no span tagger was fitted: a classifier over a few
hundred documents scored 0.754 class accuracy where the spans scored 0.907 F1.


In [ ]:
#| export
#: How learned and baseline spans are combined.
MODES = ('defer', 'union', 'hybrid', 'learned', 'baseline')

def _sk(what:str='Fitting a PII detector'):
    "scikit-learn, with an error that names the extra that fixes it."
    try:
        import sklearn; return sklearn
    except ImportError:
        raise ImportError(f"{what} needs scikit-learn. Install it with: pip install 'anya[pii]'") from None

class PiiDetector:
    "Spans and a document verdict: learned where there were examples, arithmetic where there were none."
    def __init__(self,
                 tagger:dict=None,   # the fitted span tagger, from `fit`
                 doc=None,           # the fitted document classifier, from `fit`
                 kinds=(),           # span kinds the tagger was trained on: the ones it may veto
                 base_kinds=(),      # span kinds the baseline produced while fitting: the ones it keeps
                 mode:str='defer',   # how learned and baseline spans are combined
                 bias:float=0.0,     # added to `O` when decoding: up is precision, down is recall
                 base=None,          # the fallback detector; None -> `baseline()`
                 window:int=2,       # tokens of context the tagger was fitted with
                 use_doc:bool=None,  # let the document class decide `has_pii`; None -> only with no tagger
                 meta:dict=None,     # what the fit was, for the record
                ):
        if mode not in MODES: raise ValueError(f'unknown mode {mode!r}; one of {MODES}')
        store_attr()
        self.kinds, self.base_kinds, self.meta = L(kinds), L(base_kinds), AttrDict(meta or {})

    @property
    def fitted(self) -> bool:
        "Whether anything was learned. `False` means every answer comes from the baseline."
        return self.tagger is not None or self.doc is not None

    def __repr__(self):
        if not self.fitted: return f'PiiDetector(unfitted, baseline={baseline(self.base).__name__})'
        return (f'PiiDetector(mode={self.mode}, kinds={len(self.kinds)}, '
                f'deferred={len(self.base_kinds)}, doc={"yes" if self.doc is not None else "no"}, '
                f'bias={self.bias:g})')

    def base_spans(self, text:str) -> list:
        "What the fallback detector finds, kinds normalised."
        return [(int(s), int(e), kind(k), str(v)) for s, e, k, v in baseline(self.base)(str(text or ''))]

    def learned_spans(self, text:str) -> list:
        "What the tagger finds, `[]` when nothing was fitted."
        if self.tagger is None: return []
        toks = tokens(text)
        if not toks: return []
        bt = None if 'base' not in self.tagger['used'] else base_tags(toks, self.base_spans(text))
        X = self.tagger['vec'].transform([token_feats(toks, i, bt, self.window) for i in range(len(toks))])
        lp = self.tagger['clf'].predict_log_proba(X)
        tags = viterbi(lp, self.tagger['tags'], self.tagger['trans'], self.bias)
        return [(s, e, k, str(text)[s:e]) for s, e, k in bio_spans(toks, tags)]

    def _spans(self, text:str) -> list:
        "`(start, end, kind, text, source)`, with `source` naming the layer that found it."
        text = str(text or '')
        learned = [] if self.mode == 'baseline' else self.learned_spans(text)
        if self.mode == 'learned': base = []
        elif self.tagger is None or self.mode == 'baseline': base = self.base_spans(text)
        elif self.mode == 'defer':
            base = self.base_spans(text)
            learned = [s for s in learned if s[2] not in self.base_kinds]
        else:
            base = [b for b in self.base_spans(text) if not any(b[0] < s[1] and s[0] < b[1] for s in learned)]
            if self.mode == 'hybrid': base = [b for b in base if b[2] not in self.kinds]
        return no_overlap([(*s, 'learned') for s in learned] + [(*b, 'baseline') for b in base])

    def spans(self, text:str) -> L:
        "Every span as `(start, end, kind, text)`, the same shape `vishalakshi.pii.pii_spans` returns."
        return L((s, e, k, v) for s, e, k, v, _ in self._spans(text))

    def label(self, text:str) -> AttrDict:
        "The document class and how sure the classifier is. `label` is None when none was fitted."
        if self.doc is None: return AttrDict(label=None, score=None, probs={})
        p = self.doc.predict_proba([str(text or '')])[0]
        cs, i = list(self.doc.classes_), int(np.argmax(p))
        return AttrDict(label=str(cs[i]), score=round(float(p[i]), 4),
                        probs={str(c): round(float(x), 4) for c, x in zip(cs, p)})

    def verdict(self, spans:list, label:str=None) -> bool:
        "Whether this is somebody's business: any span, and the document class when it may speak."
        use = self.tagger is None if self.use_doc is None else self.use_doc
        return bool(spans) or bool(use and label is not None and label not in CLEAN)

    def report(self, text:str) -> AttrDict:
        "What is in this text, whether it is somebody's business, and which layer said so."
        sp = self._spans(text)
        counts = {}
        for _, _, k, _, _ in sp: counts[k] = counts.get(k, 0) + 1
        d = self.label(text)
        n = len(str(text or ''))
        return AttrDict(has_pii=self.verdict(sp, d.label),
                        kinds=counts, n=len(sp), scanned=n, density=round(1000*len(sp)/max(n, 1), 3),
                        label=d.label, label_score=d.score, fitted=self.fitted, mode=self.mode,
                        spans=[dict(start=s, end=e, kind=k, text=v, source=src) for s, e, k, v, src in sp])

    def redact(self, text:str, mask:str=None) -> str:
        "Mask every span. `[KIND]` unless `mask` says otherwise."
        out = str(text or '')
        for s, e, k, _ in sorted(self.spans(out), reverse=True):
            out = out[:s] + (mask if mask is not None else f'[{k.upper()}]') + out[e:]
        return out

    __call__ = report


In [ ]:
#| export
def viterbi(lp:np.ndarray,     # log probabilities, one row per token
            tags:list,         # tag names, in the columns' order
            trans:np.ndarray,  # `(len(tags)+1, len(tags))`, the last row the start scores
            bias:float=0.0,    # added to `O`: up is precision, down is recall
           ) -> list:
    "The best tag path under the BIO constraints, which greedy argmax does not respect."
    n, m = lp.shape
    lp = lp.copy(); lp[:, tags.index('O')] += bias
    dp, bp = lp[0] + trans[m], np.zeros((n, m), int)
    for i in range(1, n):
        sc = dp[:, None] + trans[:m] + lp[i][None, :]
        bp[i], dp = sc.argmax(0), sc.max(0)
    path = [int(dp.argmax())]
    for i in range(n - 1, 0, -1): path.append(int(bp[i][path[-1]]))
    return [tags[i] for i in reversed(path)]

def transitions(seqs:list, tags:list) -> np.ndarray:
    "Log transition scores from tag bigrams, with the moves BIO forbids set to `-inf`."
    ix, m = {t: i for i, t in enumerate(tags)}, len(tags)
    c = np.ones((m + 1, m))
    for s in seqs:
        c[m, ix[s[0]]] += 1
        for a, b in zip(s, s[1:]): c[ix[a], ix[b]] += 1
    tr = np.log(c/c.sum(1, keepdims=True))
    for j, t in enumerate(tags):
        if not t.startswith('I-'): continue
        tr[m, j] = -1e9                                    # a document cannot open on a continuation
        for i, p in enumerate(tags):
            if p not in (t, 'B-' + t[2:]): tr[i, j] = -1e9
    return tr


## Fitting

Two heads off the same data. The span tagger is a logistic regression over token features decoded
with Viterbi; the document classifier is char and word n-grams next to the baseline's own counts.
Neither is deep and that is the point: a few hundred labelled examples is what a team actually has,
and it is enough for a linear model over the right features.

Below `min_examples` nothing is fitted and you get the baseline back, so the caller never has to ask.


In [ ]:
#| export
#: Buckets baseline kinds are hashed into for the document classifier, so its features carry no fitted state.
BASE_BUCKETS = 32

def base_matrix(texts, base=None) -> np.ndarray:
    "Document features from the baseline: hashed kind counts, how many spans, how dense, how long."
    from zlib import crc32
    b = baseline(base)
    out = np.zeros((len(texts), BASE_BUCKETS + 4))
    for i, t in enumerate(texts):
        t = str(t or ''); sp = b(t)
        for s, e, k, *_ in sp: out[i, crc32(str(k).encode()) % BASE_BUCKETS] += 1
        out[i, BASE_BUCKETS:] = [len(sp), 1000*len(sp)/max(len(t), 1), len(t)/1000,
                                 max((e - s for s, e, *_ in sp), default=0)]
    return out

def doc_pipe(C:float=4.0, max_features:int=200_000, base=None, class_weight:str='balanced', seed:int=0):
    "The document classifier: char and word n-grams beside the baseline's counts, into a linear model."
    _sk('Fitting a document classifier')
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import FeatureUnion, Pipeline, make_pipeline
    from sklearn.preprocessing import FunctionTransformer, MaxAbsScaler
    return Pipeline([('feats', FeatureUnion([
        ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True,
                                 max_features=max_features)),
        ('word', TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
        ('base', make_pipeline(FunctionTransformer(base_matrix, validate=False, kw_args={'base': base}),
                               MaxAbsScaler()))])),
        ('clf', LogisticRegression(C=C, max_iter=3000, class_weight=class_weight, random_state=seed))])


In [ ]:
#| export
def fit(data,                      # examples: a file, a folder, a `Dataset`, or an iterable
        spans:bool=True,           # fit the span tagger, when the data carries spans
        doc:bool=True,             # fit the document classifier, when the data carries two classes
        mode:str='defer',          # how learned and baseline spans are combined
        C:float=4.0,               # inverse regularisation, both heads
        class_weight:str='balanced',  # `O` outnumbers every tag ten to one, so correct the prior
        window:int=2,              # tokens of context each side
        base=None,                 # the fallback detector; None -> `baseline()`
        base_feats:bool=True,      # let the tagger see what the baseline said
        min_examples:int=8,        # below this nothing is fitted and the baseline is the detector
        max_features:int=200_000,  # cap on the document classifier's n-gram vocabulary
        bias:float=0.0,            # decoding bias; `tune_bias` picks one
        seed:int=0,
       ) -> PiiDetector:
    "Fit a detector on your examples. Too few to learn from and you get the baseline detector back."
    ds = dataset(data)
    meta = AttrDict(n=len(ds), counts=dict(ds.counts()), mode=mode, C=C, window=window, seed=seed,
                    class_weight=class_weight,
                    base=baseline(base).__name__, base_feats=base_feats,
                    sklearn=None, fitted_at=time.strftime('%Y-%m-%d %H:%M:%S'))
    if len(ds) < min_examples:
        meta.skipped = f'{len(ds)} examples, under min_examples={min_examples}'
        return PiiDetector(mode=mode, base=base, window=window, bias=bias, meta=meta)
    b = baseline(base)
    bk = sorted({k for e in ds for _, _, k, *_ in b(e.text)})
    meta.base_kinds = bk
    meta.sklearn = _sk().__version__
    tagger = _fit_tagger(ds, base, base_feats, window, C, class_weight, seed) if spans else None
    docm = _fit_doc(ds, base, C, class_weight, max_features, seed) if doc else None
    if tagger is None and docm is None:
        meta.skipped = 'no spans to tag and fewer than two document classes'
    return PiiDetector(tagger=tagger, doc=docm, kinds=(tagger or {}).get('kinds', ()), base_kinds=bk,
                       mode=mode, bias=bias, base=base, window=window, meta=meta)

def _fit_tagger(ds, base, base_feats, window, C, class_weight, seed):
    "The span head. Trained on the span-annotated examples plus the ones marked clean, which are all `O`."
    from sklearn.feature_extraction import DictVectorizer
    from sklearn.linear_model import LogisticRegression
    b = baseline(base)
    # a positive with no spans was never span-annotated: teaching the tagger it is all `O` is a lie
    tr = [e for e in ds if e.spans or e.label in CLEAN]
    X, y, seqs = [], [], []
    for e in tr:
        toks = tokens(e.text)
        if not toks: continue
        bt = base_tags(toks, b(e.text)) if base_feats else None
        tags = bio(toks, e.spans)
        X += [token_feats(toks, i, bt, window) for i in range(len(toks))]
        y += tags; seqs.append(tags)
    if len(set(y)) < 2: return None
    vec = DictVectorizer()
    clf = LogisticRegression(C=C, max_iter=3000, class_weight=class_weight,
                             random_state=seed).fit(vec.fit_transform(X), y)
    tags = [str(t) for t in clf.classes_]
    return dict(vec=vec, clf=clf, tags=tags, trans=transitions(seqs, tags),
                kinds=sorted({t[2:] for t in tags if t != 'O'}),
                used=['base'] if base_feats else [], n=len(tr), n_tokens=len(y))

def _fit_doc(ds, base, C, class_weight, max_features, seed):
    "The document head, when there are two classes to tell apart."
    lab = [e for e in ds if e.text]
    if len({e.label for e in lab}) < 2: return None
    return doc_pipe(C, max_features, base, class_weight, seed).fit([e.text for e in lab],
                                                                  [e.label for e in lab])


In [ ]:
#| hide
_tr, _va = dataset(FIX/'pii_org.jsonl').split(0.25, seed=0)
_det = fit(_tr, seed=0)
test_eq(_det.fitted, True)
test_eq('emp_id' in _det.kinds, True)
# the thing no pattern bank can do: an identifier this organisation made up
test_eq(first(k for _, _, k, _ in _det.spans('Escalated by EMP-774310 after the second call.')), 'emp_id')
test_eq(_det.report('The committee deferred the decision.').has_pii, False)
# the document classifier is fitted, and stays out of the verdict while the tagger is there to make it
test_eq(_det.verdict([], 'pii'), False)
test_eq(PiiDetector(doc=_det.doc).verdict([], 'pii'), True)
# `defer` never overrules the baseline, `hybrid` does, and it cuts both ways: the floor patterns
# read a Luhn-valid device serial as a card, and the veto that removes it removes a real card too
_serial, _card = ('Serial 6560976701720096 was returned under warranty.',
                  'Payment taken on card 4650300891319344 at the till.')
_hy = fit(_tr, mode='hybrid', seed=0)
test_eq([_det.report(_serial).has_pii, _hy.report(_serial).has_pii], [True, False])
test_eq([_det.report(_card).has_pii, _hy.report(_card).has_pii], [True, False])
test_eq(_det.redact('Badge B-77341 was lost.'), 'Badge [BADGE] was lost.')
test_eq(_det.label('Referral PT/2024/88213 was triaged.').label, 'pii')

# too little to learn from: the same object, answering out of the pattern bank
_un = fit([{'text': 'mail jane@example.com'}])
test_eq(_un.fitted, False)
test_eq(_un.spans('mail jane@example.com')[0][2], 'email')
test_eq('under min_examples' in _un.meta.skipped, True)
test_eq(repr(_un).startswith('PiiDetector(unfitted'), True)
test_fail(lambda: fit(_tr, mode='sometimes'), contains='unknown mode')


## Saving one

`joblib`, under `$ANYA_HOME/pii/models/<name>` unless you name a path. Loading one unpickles it, so
load your own and nobody else's.


In [ ]:
#| export
def pii_home() -> Path:
    "Where fitted detectors and runs live: `$ANYA_HOME/pii`, else `~/.anya/pii`."
    return Path(os.environ.get('ANYA_HOME', Path.home()/'.anya')).expanduser()/'pii'

def pii_path(name) -> Path:
    "A name is a folder under `pii_home()/models`; anything with a separator in it is a path."
    from anya.core import safe_name
    s = str(name)
    return Path(s).expanduser() if ('/' in s or '\\' in s) else pii_home()/'models'/safe_name(s)

@patch
def save(self:PiiDetector, name) -> Path:
    "Write the fitted models and a readable `meta.json`. Returns the folder."
    import joblib
    p = pii_path(name); p.mkdir(parents=True, exist_ok=True)
    joblib.dump(dict(tagger=self.tagger, doc=self.doc, kinds=list(self.kinds), mode=self.mode,
                     base_kinds=list(self.base_kinds), bias=self.bias, base=self.base,
                     window=self.window, use_doc=self.use_doc), p/'model.joblib')
    (p/'meta.json').write_text(json.dumps(dict(self.meta, kinds=list(self.kinds), mode=self.mode,
                                               bias=self.bias, fitted=self.fitted), indent=1, default=str))
    return p

def load_pii(name) -> PiiDetector:
    "A detector saved by `PiiDetector.save`. Unpickles, so load only what you wrote."
    _sk('Loading a fitted PII detector')
    import joblib
    p = pii_path(name)
    if not (p/'model.joblib').exists(): raise FileNotFoundError(
        f'no fitted detector at {p}. Fitted ones: {", ".join(fitted_pii()) or "none"}')
    d = joblib.load(p/'model.joblib')
    meta = json.loads((p/'meta.json').read_text()) if (p/'meta.json').exists() else {}
    return PiiDetector(**d, meta=meta)

def fitted_pii() -> L:
    "The names `load_pii` will take."
    d = pii_home()/'models'
    return L(sorted(x.name for x in d.iterdir() if (x/'model.joblib').exists())) if d.exists() else L()


In [ ]:
#| hide
_home = Path(mkdtemp()); os.environ['ANYA_HOME'] = str(_home)
test_eq(pii_home(), _home/'pii')
_p = _det.save('test-org')
test_eq(sorted(x.name for x in _p.iterdir()), ['meta.json', 'model.joblib'])
test_eq(fitted_pii(), ['test-org'])
_back = load_pii('test-org')
test_eq(_back.kinds, _det.kinds)
test_eq(_back.spans('Escalated by EMP-774310 today.'), _det.spans('Escalated by EMP-774310 today.'))
test_fail(lambda: load_pii('nope'), contains='no fitted detector')


## Measuring one

Spans are scored by exact offsets and kind, and again `relaxed=True` where an overlap is enough.
Documents are scored on the has-PII decision a retrieval gate actually makes, so an unfitted detector
scores on the same scale as a fitted one and the two are comparable.

`compare` is a paired bootstrap over documents: the same resampled documents scored by both systems,
2.5th to 97.5th percentile of the difference. An interval spanning zero is no difference.


In [ ]:
#| export
def fbeta(p:float, r:float, beta:float=1.0) -> float:
    "The weighted harmonic mean. `beta` above 1 favours recall, below 1 favours precision."
    b2 = beta*beta
    return (1 + b2)*p*r/(b2*p + r) if (b2*p + r) else 0.0

def prf(tp:int, fp:int, fn:int, beta:float=1.0) -> AttrDict:
    "Precision, recall and F-beta off three counts."
    p = tp/(tp + fp) if tp + fp else 0.0
    r = tp/(tp + fn) if tp + fn else 0.0
    return AttrDict(tp=tp, fp=fp, fn=fn, precision=round(p, 4), recall=round(r, 4),
                    f1=round(fbeta(p, r, beta), 4))

def match_spans(gold:list, pred:list, relaxed:bool=False) -> tuple:
    "`(matched, missed, spurious)`. Exact wants the same offsets and kind; relaxed wants an overlap."
    same = (lambda g, p: g[0] < p[1] and p[0] < g[1]) if relaxed else (lambda g, p: g[:2] == p[:2])
    used, hit, miss = set(), [], []
    for g in gold:
        j = first(j for j, p in enumerate(pred) if j not in used and p[2] == g[2] and same(g, p))
        if j is None: miss.append(g)
        else: used.add(j); hit.append((g, pred[j]))
    return hit, miss, [p for j, p in enumerate(pred) if j not in used]


In [ ]:
#| export
def evaluate(det:PiiDetector,        # what to score
             data,                   # held-out examples
             relaxed:bool=False,     # count an overlapping span of the right kind as a hit
             beta:float=1.0,         # what F favours
             max_errors:int=200,     # how many mistakes to keep for the report
            ) -> AttrDict:
    "Span and document scores on held-out examples, with the mistakes that produced them."
    ds, per, errs = dataset(data), [], []
    kc = {}
    t0 = time.perf_counter()
    for i, e in enumerate(ds):
        sp = det._spans(e.text)
        hit, miss, spur = match_spans(e.spans, [(s, en, k) for s, en, k, _, _ in sp], relaxed)
        for g in e.spans: kc.setdefault(g[2], [0, 0, 0])[2] += 1        # support counted as fn, fixed below
        for g, _ in hit: kc[g[2]][0] += 1; kc[g[2]][2] -= 1
        for p in spur: kc.setdefault(p[2], [0, 0, 0])[1] += 1
        want, lab = bool(e.spans) or e.label not in CLEAN, det.label(e.text)
        got = AttrDict(has_pii=det.verdict(sp, lab.label), label=lab.label)
        per.append(dict(tp=len(hit), fp=len(spur), fn=len(miss),
                        dtp=int(want and got.has_pii), dfp=int(got.has_pii and not want),
                        dfn=int(want and not got.has_pii), dok=int(got.has_pii == want),
                        cok=int(got.label == e.label) if got.label is not None else None))
        if len(errs) < max_errors:
            for g in miss: errs.append(_error(i, e, 'missed', g))
            for p in spur: errs.append(_error(i, e, 'spurious', p))
            if got.has_pii != want: errs.append(_error(i, e, 'doc_' + ('fp' if got.has_pii else 'fn'), None))
    ms = 1000*(time.perf_counter() - t0)/max(len(ds), 1)
    tot = lambda k: sum(d[k] for d in per)
    cok = [d['cok'] for d in per if d['cok'] is not None]
    return AttrDict(
        n=len(ds), ms_per_doc=round(ms, 3), relaxed=relaxed, mode=det.mode, bias=det.bias,
        fitted=det.fitted, kinds=list(det.kinds),
        spans=prf(tot('tp'), tot('fp'), tot('fn'), beta),
        by_kind={k: dict(prf(v[0], v[1], v[2], beta), support=v[0] + v[2]) for k, v in sorted(kc.items())},
        doc=dict(prf(tot('dtp'), tot('dfp'), tot('dfn'), beta),
                 accuracy=round(sum(d['dok'] for d in per)/max(len(per), 1), 4),
                 tn=sum(1 for d in per if not d['dtp'] and not d['dfp'] and not d['dfn'])),
        class_accuracy=round(sum(cok)/len(cok), 4) if cok else None,
        per_doc=per, errors=errs)

def _error(i, e, why, span) -> dict:
    "One mistake, with enough of the text around it to judge by eye."
    s, en, k = span if span else (0, 0, '')
    lo, hi = max(0, s - 60), min(len(e.text), en + 60)
    return dict(i=i, why=why, kind=k, text=e.text[s:en], label=e.label,
                before=e.text[lo:s], after=e.text[en:hi], full=e.text[:400])


In [ ]:
#| export
def tune_bias(det:PiiDetector,   # fitted, and modified in place
              data,              # a split the model was not fitted on
              biases=None,       # what to try; None -> -4 to 8
              beta:float=1.0,    # above 1 favours recall
              relaxed:bool=False,
             ) -> AttrDict:
    "Pick the decoding bias with the best F-beta on `data`, and set it. Returns the sweep."
    if det.tagger is None: return AttrDict(best=det.bias, sweep=[])
    bs = list(np.arange(-4, 8.5, 0.5)) if biases is None else list(biases)
    was, sweep = det.bias, []
    for b in bs:
        det.bias = float(b)
        m = evaluate(det, data, relaxed=relaxed, beta=beta, max_errors=0)
        sweep.append(dict(bias=round(float(b), 3), **{k: m.spans[k] for k in ('precision', 'recall', 'f1')}))
    best = max(sweep, key=lambda r: (r['f1'], -abs(r['bias'])))
    det.bias = best['bias'] if sweep else was
    return AttrDict(best=det.bias, beta=beta, sweep=sweep)

def counts_of(per_doc:list, level:str='spans') -> np.ndarray:
    "The `(tp, fp, fn)` per document that a bootstrap resamples, from `evaluate`."
    ks = ('tp', 'fp', 'fn') if level == 'spans' else ('dtp', 'dfp', 'dfn')
    return np.array([[d[k] for k in ks] for d in per_doc], float).reshape(-1, 3)

def bootstrap(A:np.ndarray,      # `counts_of` for one system
              B:np.ndarray,      # the same documents, in the same order, for another
              n:int=1000,        # resamples
              seed:int=0,
             ) -> AttrDict:
    "Paired bootstrap over documents: F1 of `B` minus F1 of `A`, and the interval it sits in."
    f1 = lambda m: prf(*m.sum(0).astype(int)).f1 if len(m) else 0.0
    rng = np.random.default_rng(seed)
    deltas = [f1(B[ix]) - f1(A[ix]) for ix in (rng.integers(0, len(A), len(A)) for _ in range(n))] \
        if len(A) else [0.0]
    lo, hi = (float(x) for x in np.percentile(deltas, [2.5, 97.5]))
    return AttrDict(a=f1(A), b=f1(B), delta=round(f1(B) - f1(A), 4), lo=round(lo, 4), hi=round(hi, 4),
                    n_boot=n, difference=not (lo <= 0 <= hi))

def compare(a:PiiDetector,       # one detector
            b:PiiDetector,       # another, on the same data
            data,                 # held out from both
            level:str='spans',    # 'spans' or 'doc'
            n:int=1000,           # bootstrap resamples
            seed:int=0,
            relaxed:bool=False,
           ) -> AttrDict:
    "Paired bootstrap over documents: the F1 difference `b - a`, and the interval it sits in."
    ea, eb = (evaluate(d, data, relaxed=relaxed, max_errors=0) for d in (a, b))
    return AttrDict(level=level, **bootstrap(counts_of(ea.per_doc, level),
                                             counts_of(eb.per_doc, level), n, seed))


In [ ]:
#| hide
_m = evaluate(_det, _va)
test_eq(_m.n, len(_va))
test_eq(_m.spans.f1 > 0.9, True)
# what `defer` guarantees: every false positive left is one the baseline made, none is invented
test_eq({k for k, v in _m.by_kind.items() if v['fp']} <= set(_det.base_kinds), True)
test_eq(set(_m.by_kind) <= set(_ds.kinds) and len(_m.by_kind) > 3, True)
# support is what was in the gold, so it survives whatever the model did
test_eq(sum(v['support'] for v in _m.by_kind.values()), sum(len(e.spans) for e in _va))
test_eq(match_spans([(0, 3, 'k')], [(1, 3, 'k')]), ([], [(0, 3, 'k')], [(1, 3, 'k')]))
test_eq(len(match_spans([(0, 3, 'k')], [(1, 3, 'k')], relaxed=True)[0]), 1)
test_eq(prf(1, 1, 2), dict(tp=1, fp=1, fn=2, precision=0.5, recall=0.3333, f1=0.4))
test_eq(fbeta(0, 0), 0.0)

_sw = tune_bias(fit(_tr, seed=0), _va)
test_eq(len(_sw.sweep) > 10, True)
_cmp = compare(PiiDetector(mode='baseline'), _det, _va, n=200)
test_eq(_cmp.delta > 0, True)                       # the fitted model finds what no pattern can
test_eq(_cmp.difference, True)


## One command per run

`fit_run` is the whole loop: split by group, fit, sweep the decoding bias on a split held out of the
training half, score on the validation half, compare against the baseline, and write the run to disk
with a report you can open. In a notebook the returned run renders as the report.

The sweep is measured and drawn, not adopted. Taking its argmax cost 0.064 F1 on average over five
splits of `evals/mkpii.py`: 34 dev examples do not settle where the operating point belongs, and
balanced class weights already put it somewhere sensible. `tune=True` adopts it anyway, for a corpus
big enough that the choice transfers.


In [ ]:
#| export
def fit_run(data,                 # examples: a file, a folder, a `Dataset`, or an iterable
            name:str=None,        # what to call the run; None -> a timestamp
            valid:float=0.25,     # share of groups held out for scoring
            sweep:bool=True,      # measure the bias sweep, for the report to draw
            tune:bool=False,      # and adopt its best bias; it cost 0.064 F1 on `evals/mkpii.py`
            beta:float=1.0,       # what the sweep is scored for; above 1 favours recall
            relaxed:bool=False,   # score an overlapping span of the right kind as a hit
            seed:int=0,
            dest=None,            # where runs are written; None -> `pii_home()/runs`
            save_as:str=None,     # also save the fitted detector under this name
            wandb:bool=False,     # also log the run to wandb, when it is installed
            **kw                  # forwarded to `fit`
           ):
    "Fit, score, compare against the baseline, and write the run and its report. Returns the `Run`."
    from anya.runs import new_run, save_run
    ds = dataset(data)
    tr, va = ds.split(valid, seed)
    if not len(va): va, sweep = tr, False                 # nothing held out: report on what was fitted
    sw = []
    if sweep or tune:
        tr2, dev = tr.split(0.2, seed)
        if len(dev):
            s = tune_bias(fit(tr2, seed=seed, **kw), dev, beta=beta, relaxed=relaxed)
            sw = s.sweep
            if tune: kw['bias'] = s.best
    det = fit(tr, seed=seed, **kw)
    m = evaluate(det, va, relaxed=relaxed, beta=beta)
    base = PiiDetector(mode='baseline', base=kw.get('base'))
    cmp = dict(spans=dict(compare(base, det, va, 'spans', seed=seed, relaxed=relaxed)),
               doc=dict(compare(base, det, va, 'doc', seed=seed, relaxed=relaxed)))
    cfg = dict(det.meta, name=name, n=len(ds), n_train=len(tr), n_valid=len(va), valid=valid,
               seed=seed, beta=beta, relaxed=relaxed, tuned=bool(tune and sw), source=str(data)[:200])
    r = save_run(new_run(name, dest), config=cfg, metrics=dict(m), sweep=sw, compare=cmp,
                 baseline=dict(evaluate(base, va, relaxed=relaxed, beta=beta, max_errors=0)))
    if save_as: r.saved_to = str(det.save(save_as))
    if wandb:
        from anya.runs import log_wandb
        r.wandb = log_wandb(r)
    r.detector = det
    return r


In [ ]:
#| hide
_run = fit_run(FIX/'pii_org.jsonl', name='test', dest=_home/'runs', seed=0)
test_eq(_run.metrics['spans']['recall'] > 0.5, True)
test_eq((Path(_run.path)/'report.html').exists(), True)
test_eq(_run.config['tuned'], False)                  # the sweep is drawn, not adopted
test_eq(len(_run.sweep) > 10, True)
test_eq(fit_run(FIX/'pii_org.jsonl', dest=_home/'runs', seed=0, tune=True).config['tuned'], True)
test_eq(_run.compare['spans']['delta'] > 0, True)


## What it is worth

`evals/pii_learned.py` measures this against the pattern bank on the corpus in `evals/mkpii.py`,
which plants five made-up identifier kinds and the lookalikes that share their shapes. Numbers, the
method behind them and the intervals are in `evals/RESULTS.md`.


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
